# Indian airlines after COVID: performance index and dashboard
Fifth notebook. Run it after the download and cleaning notebooks. It **reads** `data/clean/` and **writes one file**: `outputs/dashboard/dgca_performance_dashboard_<date>.html`, a self-contained page (Plotly is embedded, so it opens offline).

**What it adds to the project**
- **10 metrics** in three pillars (Recovery, Operations, Momentum), each measured over the latest 12 months.
- **A 0 to 100 performance index** for every airline, plus three tests of whether it means anything: does it predict the next six months, would it have flagged Go First, and does the ranking survive different weightings?
- **28 charts, each a different chart form**: gauges, table, radar, heatmap, parallel coordinates, bump, slope, sunburst, treemap, Sankey, waterfall, ternary, Marimekko, funnel, polar bar, 3D surface, candlestick, fan chart, box, violin, histogram with rug, density contour, scatter matrix, dendrogram, bubble, 3D scatter, lollipop and range dot plot.
- **Conclusions** whose numbers are computed, so they stay correct if the data is re-collected.

Libraries: pandas, numpy, plotly, scipy, pyarrow (`pip install plotly scipy pyarrow`).

## 1. Setup and data

In [1]:
# SETUP: paths, libraries, data. This notebook only READS data/ and only WRITES outputs/dashboard/.
import warnings; warnings.filterwarnings("ignore", category=FutureWarning)
import calendar, html as _html
from pathlib import Path
import numpy as np, pandas as pd
import plotly.graph_objects as go
import plotly.figure_factory as ff
from plotly.subplots import make_subplots
from plotly.colors import sample_colorscale
from plotly.offline import get_plotlyjs
from scipy.cluster.hierarchy import linkage, fcluster
from IPython.display import display, Markdown

SHOW_FIGURES = True        # False = only build the HTML file (faster, smaller notebook)
TODAY = pd.Timestamp.today().date().isoformat()
_here = Path.cwd().resolve()
PROJECT_ROOT = next(p for p in [_here, *_here.parents] if (p / "data" / "clean").is_dir())
CLEAN = PROJECT_ROOT / "data" / "clean"
OUT = PROJECT_ROOT / "outputs" / "dashboard"; OUT.mkdir(parents=True, exist_ok=True)
latest = lambda folder, pat: sorted(folder.glob(pat))[-1]

clean = pd.read_parquet(latest(CLEAN, "dgca_domestic_clean_*.parquet"))
rec = pd.read_csv(latest(CLEAN, "total_domestic_reconciliation_*.csv"), parse_dates=["Date"])
LAST = rec.loc[rec.within_2pct, "Date"].max()          # last month where the airlines add up to DGCA's own total
MIN_PAX = 25_000            # an airline is scored only if it averaged at least 25,000 passengers a month over the window
MIN_BASE_SHARE = 0.005      # 2019 comparisons need at least 0.5% of 2019 passengers (same rule as the report notebook)
print(f"clean rows: {len(clean):,} | analysis ends: {LAST:%B %Y}")


clean rows: 1,957 | analysis ends: August 2026


## 2. Monthly matrices per airline group
One column per airline group (Air India Group combines Air India, Air India Express, Vistara, AirAsia India and AIX Connect). April and May 2020 are set to zero for airlines that reported before and after, because nobody flew.

In [2]:
# GROUP-MONTH MATRICES for scheduled domestic traffic (one column per airline group)
sd = clean[(clean.Type == "ScheduledDomestic") & (clean.Date <= LAST)]
months = pd.date_range("2017-01-01", LAST, freq="MS")
FIELDS = dict(pax="Passenger Number", rpk="Passenger Kilometers", ask="Seat Kilometers",
              dep="Aircraft Number", hrs="Aircraft Hours")      # 'Aircraft Number' = departures (DGCA header: DEPARTURES)
P = {k: sd.groupby(["Group", "Date"])[c].sum(min_count=1).unstack("Group").reindex(months).astype(float) for k, c in FIELDS.items()}
# April and May 2020: nobody flew, so nothing was reported. Treated as a structural zero for airlines that reported before and after.
for k in P:
    for g in P[k].columns:
        if pd.notna(P[k].loc["2020-03-01", g]) and pd.notna(P[k].loc["2020-06-01", g]):
            for m in ("2020-04-01", "2020-05-01"):
                if pd.isna(P[k].loc[m, g]): P[k].loc[m, g] = 0.0
mk = P["pax"].sum(axis=1)                                      # whole-market passengers per month

b19i = months[months.year == 2019]
b19 = {k: P[k].loc[b19i] for k in P}
b19m = b19["pax"].set_axis(b19i.month)                         # 2019 passengers indexed by month number
share19 = b19["pax"].sum() / b19["pax"].sum().sum()
BASE = [g for g in P["pax"].columns if (b19["pax"][g] > 0).sum() == 12 and share19[g] >= MIN_BASE_SHARE]   # groups with a fair 2019 yardstick
RATIO = pd.DataFrame({g: P["pax"][g] / pd.Series(months.month, index=months).map(b19m[g]) for g in BASE})   # passengers vs same month of 2019
ROLL = RATIO.rolling(3).mean()

def status_of(g):
    tr = P["pax"][g][P["pax"][g] > 0]
    if tr.empty: return "No traffic"
    operating = tr.index.max() == LAST
    if g in BASE:
        hit = ROLL.loc["2020-07-01":, g].dropna(); hit = hit[hit >= 1]
        if len(hit): return "Recovered and holding" if (operating and RATIO[g].iloc[-3:].mean() >= 1) else "Recovered, then fell back"
        return "Never recovered (still flying)" if operating else "Never recovered (stopped flying)"
    if b19["pax"][g].sum() == 0: return "New entrant (no 2019 level)"
    return "Not tested (partial or tiny 2019 base)"
STATUS = {g: status_of(g) for g in P["pax"].columns}
print("groups with a fair 2019 yardstick:", BASE)
print(pd.Series(STATUS)[lambda s: s != "No traffic"].value_counts().to_string())


groups with a fair 2019 yardstick: ['Air India Group', 'Alliance Air', 'Go First', 'IndiGo', 'SpiceJet', 'TruJet']
New entrant (no 2019 level)               10
Not tested (partial or tiny 2019 base)     4
Recovered and holding                      2
Never recovered (stopped flying)           2
Recovered, then fell back                  1
Never recovered (still flying)             1


## 3. The ten metrics and the performance index
| Pillar | Metric | What it captures |
|---|---|---|
| Recovery | Demand recovery, capacity recovery, load-factor change vs 2019, consistency vs 2019 | How far each airline is from its own 2019 level (only airlines with a full, non-tiny 2019) |
| Operations | Load factor, passengers per departure, network productivity | How well the aircraft and seats are used |
| Momentum | Growth momentum, share momentum, volatility | Where the airline is heading |

**Scoring.** Each metric becomes a score: 50 + 25 x (how many standard deviations the airline is from the peer average), capped at 0 and 100. Volatility is flipped (lower is better). A pillar is the average of its metric scores, and the **index** is the average of the pillars the airline has. The index is computed for every month since June 2023, which the bump chart and the prediction tests use.

In [3]:
# THE 10 METRICS, measured "as of" a month T over the 12 months ending T (and the 12 months before, for momentum)
MET = {   # key: (label, unit, higher is better?, pillar)
    "dem":  ("Demand recovery",            "% of 2019 passenger-km",               True,  "Recovery"),
    "cap":  ("Capacity recovery",          "% of 2019 seat-km",                    True,  "Recovery"),
    "lfd":  ("Load-factor change vs 2019", "percentage points",                    True,  "Recovery"),
    "con":  ("Consistency vs 2019",        "% of months at or above 2019",         True,  "Recovery"),
    "lf":   ("Load factor",                "% of seat-km filled",                  True,  "Operations"),
    "ppd":  ("Passengers per departure",   "passengers",                           True,  "Operations"),
    "prod": ("Network productivity",       "thousand pax-km per aircraft hour",    True,  "Operations"),
    "gro":  ("Growth momentum",            "% vs previous 12 months",              True,  "Momentum"),
    "shm":  ("Share momentum",             "pp change in market share",            True,  "Momentum"),
    "vol":  ("Volatility",                 "std. dev. of monthly YoY growth (pp)", False, "Momentum"),
}
WHY = {
    "dem":  "Distance-weighted demand against the pre-COVID yardstick. Passenger-km also reflects how far people fly, not just how many.",
    "cap":  "Seat-km is what the airline chose to supply. Rising capacity shows confidence and funding; a cut shows retreat.",
    "lfd":  "Whether planes fill as well as in 2019. Capacity can return faster than demand, which shows up here first.",
    "con":  "How many of the last 12 months beat the same month of 2019. Separates a steady recovery from a one-month spike.",
    "lf":   "Seats filled. Low values mean fixed costs are spread over fewer paying passengers.",
    "ppd":  "Head-count per flight: aircraft size times fullness. A proxy for how much each departure earns.",
    "prod": "Passenger-km produced per aircraft hour: how hard the aircraft work. Asset productivity.",
    "gro":  "Passengers vs the previous 12 months. Growth tends to continue, so it is the natural leading indicator.",
    "shm":  "Market share gained or lost over a year, in percentage points. Shows who wins customers from rivals.",
    "vol":  "Spread of monthly year-on-year growth. Erratic traffic means weaker planning and revenue. Lower is better.",
}
PILLARS = ["Recovery", "Operations", "Momentum"]
b19_lf = {g: b19["rpk"][g].sum() / b19["ask"][g].sum() * 100 for g in P["pax"].columns if b19["ask"][g].sum() > 0}

def metrics_asof(T):
    T = pd.Timestamp(T); i = months.get_loc(T)
    if i < 11: return pd.DataFrame(columns=list(MET))
    W = months[i - 11:i + 1]; PW = months[i - 23:i - 11] if i >= 23 else None
    rows = {}
    for g in P["pax"].columns:
        pw = P["pax"].loc[W, g]
        if not ((pw > 0).sum() == 12 and pw.mean() >= MIN_PAX): continue
        s = lambda k: P[k].loc[W, g].sum()
        r = dict(lf=s("rpk") / s("ask") * 100, ppd=s("pax") / s("dep"), prod=s("rpk") / s("hrs"))
        if PW is not None and (P["pax"].loc[PW, g] > 0).sum() == 12:
            prev = P["pax"].loc[PW, g].sum()
            r["gro"] = (pw.sum() / prev - 1) * 100
            r["shm"] = (pw.sum() / mk.loc[W].sum() - prev / mk.loc[PW].sum()) * 100
            r["vol"] = np.std((pw.values / P["pax"].loc[PW, g].values - 1) * 100)
        if g in BASE:
            r["dem"] = s("rpk") / b19["rpk"][g].sum() * 100
            r["cap"] = s("ask") / b19["ask"][g].sum() * 100
            r["lfd"] = r["lf"] - b19_lf[g]
            r["con"] = np.mean(pw.values >= b19m[g].reindex(W.month).values) * 100
        rows[g] = r
    return pd.DataFrame(rows).T.reindex(columns=list(MET)).astype(float)

def score(M):
    """Metric score = 50 + 25 x z, where z compares an airline with the other airlines scored that month
    (capped at +/-2, so 0 to 100; 50 = peer average). Lower-is-better metrics are flipped.
    Pillar = mean of its available metric scores. Index = mean of the available pillars."""
    S = pd.DataFrame(np.nan, index=M.index, columns=M.columns)
    for k in M.columns:
        x = M[k].dropna()
        if len(x) < 3: continue
        z = ((x - x.mean()) / x.std(ddof=0)).clip(-2, 2).fillna(0)
        S.loc[x.index, k] = 50 + 25 * (z if MET[k][2] else -z)
    PIL = pd.DataFrame({p: S[[k for k in MET if MET[k][3] == p]].mean(axis=1) for p in PILLARS})
    PIL["Index"] = PIL[PILLARS].mean(axis=1)
    return S, PIL

def reweigh(PIL, w):   # pillar weights; pillars an airline lacks are dropped and the rest re-scaled
    w = pd.Series(w); ok = PIL[PILLARS].notna()
    return (PIL[PILLARS].fillna(0) * w).sum(axis=1) / (ok * w).sum(axis=1)

M0 = metrics_asof(LAST); S0, PIL0 = score(M0)
ORDER = PIL0.sort_values("Index", ascending=False).index.tolist()
HIST = {T: (lambda m: (m,) + score(m))(metrics_asof(T)) for T in months[months >= "2023-06-01"]}   # the index at every month since mid-2023
print(f"airlines scored at {LAST:%b %Y}:", ORDER)
PIL0.loc[ORDER, PILLARS + ["Index"]].round(1)


airlines scored at Aug 2026: ['Akasa Air', 'IndiGo', 'Air India Group', 'SpiceJet', 'Star Air', 'Fly91', 'Alliance Air']


,Recovery,Operations,Momentum,Index
Akasa Air,NaN,80.3,64.6,72.5
IndiGo,71.6,69.0,57.0,65.9
Air India Group,74.8,63.5,36.7,58.3
SpiceJet,21.0,68.5,51.8,47.1
Star Air,NaN,28.6,59.3,44.0
Fly91,NaN,28.0,56.6,42.3
Alliance Air,32.6,12.0,24.5,23.0


## 4. Does the index predict anything?
Three checks that only use data available at the time: (1) rank correlation of each metric with passenger growth over the following six months, (2) a backtest scoring every airline in Go First's last month, and (3) the error of a simple six-month projection rule on past months.

In [4]:
# PREDICTION TESTS: each uses only information available at the time, then looks forward
spearman = lambda a, b: a.rank().corr(b.rank())

# (1) Does a metric measured at month T line up with passenger growth over the NEXT 6 months?
#     Outcome = passengers in T+1..T+6 vs the same six months a year earlier, so seasonality cancels.
rows = []
for T, (Mt, St, Pt) in HIST.items():
    i = months.get_loc(T)
    if i + 6 >= len(months) or len(Mt) < 5: continue
    fut, prev = months[i + 1:i + 7], months[i - 11:i - 5]
    for g in Mt.index:
        y = (P["pax"].loc[fut, g].sum() / P["pax"].loc[prev, g].sum() - 1) * 100
        rows.append(dict(T=T, g=g, y=y, Index=Pt.loc[g, "Index"], **{k: Mt.loc[g, k] for k in MET}, **{p: Pt.loc[g, p] for p in PILLARS}))
OBS = pd.DataFrame(rows)
PRED = pd.Series({c: spearman(OBS[[c, "y"]].dropna()[c], OBS[[c, "y"]].dropna()["y"]) for c in list(MET) + PILLARS + ["Index"]})
for k in MET:                      # sign so that + means "the metric's better direction goes with faster growth" (volatility: lower is better)
    if not MET[k][2]: PRED[k] = -PRED[k]
NOBS = OBS[list(MET) + PILLARS + ["Index"]].notna().sum()
print(f"{len(OBS)} airline-months, {OBS['T'].min():%b %Y} to {OBS['T'].max():%b %Y}. Rank correlation with growth over the next 6 months:")
print(PRED.round(2).sort_values(ascending=False).to_string())

# (2) Backtest: score every airline flying in Go First's last month (May 2023), using only data up to then
BT_T = pd.Timestamp("2023-05-01")
BT_M = metrics_asof(BT_T); BT_S, BT_P = score(BT_M)
BT_rank = BT_P["Index"].rank(ascending=False).astype(int)
print(f"\nBacktest as of {BT_T:%b %Y}: Go First ranks {BT_rank['Go First']} of {len(BT_P)} (index {BT_P.loc['Go First', 'Index']:.1f})")

# (3) Projection rule: same month last year x (last 12 months / the 12 months before). Tested on every past origin we can check.
def forecast(s, T, h=6):
    i = s.index.get_loc(T); g = s.iloc[i - 11:i + 1].sum() / s.iloc[i - 23:i - 11].sum()
    return pd.Series([s.iloc[i + k - 12] * g for k in range(1, h + 1)])
errs = {h: [] for h in range(1, 7)}
for T in months[(months >= "2024-08-01") & (months <= months[-7])]:
    f = forecast(mk, T)
    for h in range(1, 7): errs[h].append(abs(f.iloc[h - 1] / mk.iloc[months.get_loc(T) + h] - 1))
MAPE = pd.Series({h: np.mean(e) * 100 for h, e in errs.items()})
print("\nProjection test, mean absolute % error by months ahead:", MAPE.round(1).to_dict())


190 airline-months, Jul 2023 to Feb 2026. Rank correlation with growth over the next 6 months:
gro           0.79
con           0.63
Recovery      0.59
shm           0.46
cap           0.41
dem           0.39
Momentum      0.37
ppd           0.16
Index         0.09
prod         -0.03
Operations   -0.06
lf           -0.11
vol          -0.33
lfd          -0.36

Backtest as of May 2023: Go First ranks 5 of 5 (index 36.0)

Projection test, mean absolute % error by months ahead: {1: 4.3, 2: 4.9, 3: 5.4, 4: 5.4, 5: 5.8, 6: 6.1}


## 5. Chart registry and colours
Airline colours come from a colour-blind-checked palette and follow the airline, never its rank. Lines and markers also get a distinct symbol, so charts still read in greyscale.

In [5]:
# CHART REGISTRY: each chart is stored with its section, a plain title, a "how to read" line and a caption whose numbers are computed
FIGS = {}
SEC = {1: "Scoreboard: who performs best?", 2: "Market structure: who carries the passengers?",
       3: "Time patterns: rhythm, range and projection", 4: "Distributions: how steady is each airline?",
       5: "Relationships and prediction: what predicts performance?"}
# Airline colours: validated with the dataviz palette checker (lightness, chroma, colour-blind separation all pass).
# Colour follows the airline, never its rank. Symbols add a second cue for greyscale and print.
COL = {"IndiGo": "#0072B2", "Air India Group": "#D55E00", "SpiceJet": "#E69F00", "Akasa Air": "#009E73",
       "Star Air": "#56B4E9", "Alliance Air": "#CC79A7", "Fly91": "#A0522D", "Go First": "#6A3D9A", "TruJet": "#8E44AD", "Jet Airways": "#333333"}
SYM = {"IndiGo": "circle", "Air India Group": "square", "SpiceJet": "diamond", "Akasa Air": "triangle-up",
       "Star Air": "x", "Alliance Air": "star", "Fly91": "hexagon", "Go First": "cross", "TruJet": "pentagon"}
gc = lambda g: COL.get(g, "#999999"); gs = lambda g: SYM.get(g, "circle")
VIR = lambda v: sample_colorscale("Viridis", [float(np.clip(v, 0, 100)) / 100])[0]     # luminance-ordered, so it survives greyscale
f1 = lambda v: f"{v:,.1f}"
fmt = lambda v: "" if pd.isna(v) else (f"{v:.1f}" if abs(v) < 10 else f"{v:.0f}")
def put(key, sec, title, how, fig, cap, h=470):
    m = fig.layout.margin                               # keep a chart's own margins; only fill in the ones it did not set
    fig.update_layout(template="plotly_white", font=dict(family="system-ui, 'Segoe UI', Arial, sans-serif", size=13, color="#222"), height=h,
                      margin=dict(l=60 if m.l is None else m.l, r=30 if m.r is None else m.r, t=30 if m.t is None else m.t, b=55 if m.b is None else m.b),
                      legend=dict(orientation="h", y=-0.2, x=0))
    FIGS[key] = dict(sec=sec, title=title, how=how, fig=fig, cap=cap)
COMMON = ["lf", "ppd", "prod", "gro", "shm", "vol"]          # the six metrics every scored airline has (no 2019 yardstick needed)
top, bot = ORDER[0], ORDER[-1]
FULL = PIL0.dropna(subset=["Recovery"]).sort_values("Index", ascending=False)   # airlines scored on all three pillars
W12 = months[-12:]


## 6. Section 1: scoreboard (7 charts)

In [6]:
# ---------------- SECTION 1: SCOREBOARD ----------------
# 1.1 GAUGES: the index now, with the change in standing vs 12 months earlier
PILp = HIST[LAST - pd.DateOffset(months=12)][2]
fig = make_subplots(rows=2, cols=4, specs=[[{"type": "indicator"}] * 4] * 2, vertical_spacing=0.32)
for k, g in enumerate(ORDER):
    d = dict(reference=float(PILp.loc[g, "Index"]), valueformat=".0f") if g in PILp.index else {}
    fig.add_trace(go.Indicator(mode="gauge+number" + ("+delta" if d else ""), value=float(PIL0.loc[g, "Index"]), number=dict(valueformat=".0f"), delta=d,
        title=dict(text=g, font=dict(size=14)), gauge=dict(axis=dict(range=[0, 100]), bar=dict(color=gc(g), thickness=.55),
        steps=[dict(range=[0, 40], color="#efefef"), dict(range=[40, 60], color="#e2e2e2"), dict(range=[60, 100], color="#d4d4d4")])), row=k // 4 + 1, col=k % 4 + 1)
put("gauges", 1, "How does each airline score today?", "Filled arc and number = performance index (0 to 100, 50 = peer average). The small arrow is the change since a year ago; no arrow = not scored a year ago.", fig,
    f"{top} has the highest index ({PIL0.loc[top, 'Index']:.0f}) and {bot} the lowest ({PIL0.loc[bot, 'Index']:.0f}). "
    f"Among the airlines scored on all three pillars, {FULL.index[0]} leads ({FULL.Index.iloc[0]:.0f}); airlines without a 2019 level are scored on two pillars only.", 430)

# 1.2 SCORECARD TABLE: every raw metric and the scores behind the index
THEAD = {"dem": "Demand<br>recovery", "cap": "Capacity<br>recovery", "lfd": "LF change<br>vs 2019", "con": "Consistency<br>vs 2019", "lf": "Load<br>factor",
         "ppd": "Pax per<br>departure", "prod": "Network<br>productivity", "gro": "Growth<br>momentum", "shm": "Share<br>momentum", "vol": "Volatility"}
hdr = ["#", "Airline", "Index", "Recov-<br>ery", "Oper-<br>ations", "Mom-<br>entum"] + [THEAD[k] for k in MET]
cells, fills = [[i + 1 for i in range(len(ORDER))], ORDER], [["#fff"] * len(ORDER)] * 2
for c in ["Index", "Recovery", "Operations", "Momentum"]:
    cells.append([("n/a" if pd.isna(PIL0.loc[g, c]) else f"{PIL0.loc[g, c]:.0f}") for g in ORDER])
    fills.append([("#f2f2f2" if pd.isna(PIL0.loc[g, c]) else VIR(PIL0.loc[g, c])) for g in ORDER])
for k in MET:
    cells.append([("n/a" if pd.isna(M0.loc[g, k]) else f1(M0.loc[g, k])) for g in ORDER]); fills.append(["#fff"] * len(ORDER))
fcol = [["#222"] * len(ORDER)] * 2 + [["white" if (not pd.isna(PIL0.loc[g, c]) and PIL0.loc[g, c] < 60) else "#222" for g in ORDER]
                                      for c in ["Index", "Recovery", "Operations", "Momentum"]] + [["#222"] * len(ORDER)] * len(MET)
fig = go.Figure(go.Table(columnwidth=[26, 110, 52, 60, 62, 62] + [82] * len(MET),
                         header=dict(values=hdr, fill_color="#222", font=dict(color="white", size=10.5), align="center", height=46),
                         cells=dict(values=cells, fill_color=fills, font=dict(color=fcol, size=12), align="center", height=30)))
put("table", 1, "The full scorecard", "Coloured cells are scores (yellow = strong, purple = weak). Plain cells are the raw metric values in the units listed under 'The ten metrics'. n/a = no 2019 level to compare with.", fig,
    f"{top} scores {PIL0.loc[top, 'Index']:.0f} on a load factor of {M0.loc[top, 'lf']:.0f}% and {M0.loc[top, 'ppd']:.0f} passengers per departure. "
    f"{ORDER[1]} follows at {PIL0.loc[ORDER[1], 'Index']:.0f}; {bot} is last at {PIL0.loc[bot, 'Index']:.0f} with a load factor of {M0.loc[bot, 'lf']:.0f}%.", 140 + 32 * len(ORDER) + 40)

# 1.3 RADAR: the six metrics every airline has
th = [MET[k][0] for k in COMMON]
fig = go.Figure()
for g in ORDER:
    r = [S0.loc[g, k] for k in COMMON]
    fig.add_trace(go.Scatterpolar(r=r + r[:1], theta=th + th[:1], name=g, fill="toself", opacity=.45, line=dict(color=gc(g), width=2),
                                  marker=dict(symbol=gs(g), size=8), mode="lines+markers", visible=True if g in ORDER[:4] else "legendonly"))
fig.update_layout(polar=dict(radialaxis=dict(range=[0, 100], tickvals=[25, 50, 75])))
lead = ORDER[0]
put("radar", 1, "What shape is each airline?", "Each spoke is one metric score (outer edge = best among peers). Click names in the legend to add or remove airlines.", fig,
    f"{lead} reaches {S0.loc[lead, COMMON].max():.0f} on {MET[S0.loc[lead, COMMON].idxmax()][0].lower()} but only {S0.loc[lead, COMMON].min():.0f} on {MET[S0.loc[lead, COMMON].idxmin()][0].lower()}. "
    f"No airline is strong on every spoke, which is why the index averages several metrics.", 500)

# 1.4 HEATMAP: metric scores, with the raw value printed in each cell
z = S0.loc[ORDER]; txt = [[fmt(M0.loc[g, k]) for k in MET] for g in ORDER]
fig = go.Figure(go.Heatmap(z=z.values, x=[MET[k][0] for k in MET], y=ORDER, colorscale="Viridis", zmin=0, zmax=100, text=txt, texttemplate="%{text}", xgap=2, ygap=2,
                           colorbar=dict(title=dict(text="Score")), hovertemplate="%{y}<br>%{x}<br>score %{z:.0f}, value %{text}<extra></extra>"))
fig.update_yaxes(autorange="reversed"); fig.update_xaxes(tickangle=-30)
best = S0.stack().idxmax(); worst = S0.stack().idxmin()
put("heatmap", 1, "Where is each airline strong or weak?", "Colour = score, number = the raw metric value. Blank = the airline has no 2019 level for that metric.", fig,
    f"The best single score is {best[0]} on {MET[best[1]][0].lower()} ({S0.loc[best]:.0f}); the weakest is {worst[0]} on {MET[worst[1]][0].lower()} ({S0.loc[worst]:.0f}). "
    f"Read across a row to see whether an airline is a specialist or an all-rounder.", 440)

# 1.5 PARALLEL COORDINATES: each airline is one line crossing the six metric scores
pos = {g: i for i, g in enumerate(ORDER)}
dims = [dict(label="Airline", values=[pos[g] for g in ORDER], tickvals=list(range(len(ORDER))), ticktext=ORDER, range=[-.3, len(ORDER) - .7])]
short = {"lf": "Load factor", "ppd": "Pax per dep.", "prod": "Productivity", "gro": "Growth", "shm": "Share mom.", "vol": "Volatility"}
dims += [dict(label=short[k], values=S0.loc[ORDER, k].tolist(), range=[0, 100]) for k in COMMON] + [dict(label="INDEX", values=PIL0.loc[ORDER, "Index"].tolist(), range=[0, 100])]
fig = go.Figure(go.Parcoords(line=dict(color=PIL0.loc[ORDER, "Index"].tolist(), colorscale="Viridis", cmin=0, cmax=100, showscale=True,
                                       colorbar=dict(title=dict(text="Index"))), dimensions=dims, labelangle=0, labelfont=dict(size=12)))
fig.update_layout(margin=dict(l=185, r=80, t=85, b=30))
put("parcoords", 1, "Follow each airline across the metrics", "One line per airline, coloured by its index. Axes are metric scores (0 to 100). Drag along any axis to select a range and fade out the rest.", fig,
    f"Lines that stay high across the axes belong to {top} ({PIL0.loc[top, 'Index']:.0f}) and {ORDER[1]} ({PIL0.loc[ORDER[1], 'Index']:.0f}); "
    f"{bot} drops to {S0.loc[bot, COMMON].min():.0f} on {MET[S0.loc[bot, COMMON].idxmin()][0].lower()}.", 450)

# 1.6 BUMP CHART: rank of the index month by month
H = pd.DataFrame({T: v[2]["Index"] for T, v in HIST.items()}).T          # month x airline
RK = H.rank(axis=1, ascending=False)
fig = go.Figure()
for g in RK.columns:
    s = RK[g]
    fig.add_trace(go.Scatter(x=s.index, y=s, mode="lines+markers", name=g, line=dict(color=gc(g), width=2.5), marker=dict(size=7, symbol=gs(g)), connectgaps=False,
                             hovertemplate=g + ": rank %{y:.0f} in %{x|%b %Y}<extra></extra>"))
    lv = s.dropna()
    if len(lv) and lv.index[-1] == RK.index[-1]:
        fig.add_annotation(x=lv.index[-1], y=lv.iloc[-1], text=g, xanchor="left", xshift=8, showarrow=False, font=dict(size=11, color="#222"))
fig.update_yaxes(autorange="reversed", title="Rank (1 = best)", dtick=1)
fig.update_xaxes(title="Month (index measured over the 12 months ending then)", range=[RK.index[0] - pd.Timedelta(days=20), RK.index[-1] + pd.Timedelta(days=150)])
chg = (RK.iloc[0] - RK.iloc[-1]).dropna().sort_values()
put("bump", 1, "Who has been climbing or sliding?", "Each line is an airline's rank among the scored airlines in that month. An airline joins once it has two full years of data.", fig,
    f"{chg.index[-1]} climbed {chg.iloc[-1]:.0f} place(s) since {RK.index[0]:%b %Y}, the most; {chg.index[0]} moved {chg.iloc[0]:+.0f}. "
    f"{ORDER[0]} ranks first in {int((RK[ORDER[0]] == 1).sum())} of {len(RK)} months.", 450)

# 1.7 SLOPE CHART: does the ranking survive different weightings?
W = {"Equal weights": dict(Recovery=1, Operations=1, Momentum=1), "Recovery-heavy": dict(Recovery=2, Operations=1, Momentum=1),
     "Operations-heavy": dict(Recovery=1, Operations=2, Momentum=1), "Momentum-heavy": dict(Recovery=1, Operations=1, Momentum=2)}
RKW = pd.DataFrame({n: reweigh(PIL0, w).rank(ascending=False) for n, w in W.items()})
fig = go.Figure()
for g in RKW.index:
    fig.add_trace(go.Scatter(x=list(W), y=RKW.loc[g], mode="lines+markers+text", name=g, line=dict(color=gc(g), width=2.5), marker=dict(size=11, symbol=gs(g)),
                             text=[g if j in (0, 3) else "" for j in range(4)], textposition=["middle left", "top center", "top center", "middle right"], showlegend=False))
fig.update_yaxes(autorange="reversed", dtick=1, title="Rank"); fig.update_xaxes(range=[-.7, 3.7])
swing = (RKW.max(axis=1) - RKW.min(axis=1)).sort_values(ascending=False)
put("slope", 1, "Is the ranking robust to the weighting?", "Each line follows one airline through four ways of weighting the three pillars (the heavy pillar counts double). Flat lines = a robust rank.", fig,
    f"{swing.index[0]} moves the most ({int(swing.iloc[0])} places across the four weightings); {int((swing == 0).sum())} of {len(swing)} airlines never change rank. "
    f"{RKW.idxmin()['Equal weights']} is first under equal weights and {RKW.idxmin()['Recovery-heavy']} under recovery-heavy weights.", 450)


## 7. Section 2: market structure (7 charts)

In [7]:
# ---------------- SECTION 2: MARKET STRUCTURE ----------------
br = sd[sd.Date.isin(W12)].groupby(["Group", "Airline"])[["Passenger Number"]].sum().reset_index()
br = br[(br["Passenger Number"] > 0) & br.Group.isin(ORDER)]
lf12 = {g: P["rpk"].loc[W12, g].sum() / P["ask"].loc[W12, g].sum() * 100 for g in ORDER}
share12 = br.groupby("Group")["Passenger Number"].sum() / br["Passenger Number"].sum() * 100
def hierarchy(colour):
    ids, labels, parents, values, cols = ["root"], ["Scheduled domestic"], [""], [br["Passenger Number"].sum() / 1e6], [colour("root")]
    for g in ORDER:
        d = br[br.Group == g]; kids = d.Airline.unique() if d.Airline.nunique() > 1 else []
        ids.append(g); labels.append(g); parents.append("root"); values.append(d["Passenger Number"].sum() / 1e6); cols.append(colour(g))
        for b in kids:
            ids.append(f"{g}/{b}"); labels.append(b); parents.append(g); values.append(d.loc[d.Airline == b, "Passenger Number"].sum() / 1e6); cols.append(colour(g))
    return ids, labels, parents, values, cols

# 2.1 SUNBURST: share of passengers, coloured by load factor
lf_all = P["rpk"].loc[W12, ORDER].sum().sum() / P["ask"].loc[W12, ORDER].sum().sum() * 100
ids, labels, parents, values, cols = hierarchy(lambda n: lf_all if n == "root" else lf12.get(n.split("/")[0], lf_all))
fig = go.Figure(go.Sunburst(ids=ids, labels=labels, parents=parents, values=values, branchvalues="total", marker=dict(colors=cols, colorscale="Viridis", cmin=60, cmax=95,
        colorbar=dict(title=dict(text="Load factor %"))), hovertemplate="%{label}<br>%{value:.1f} million passengers (12 months)<extra></extra>", insidetextorientation="radial"))
put("sunburst", 2, "Who carries the passengers, brand by brand?", "Inner ring = airline groups, outer ring = the brands inside them. Angle = passengers over the last 12 months; colour = load factor.", fig,
    f"{share12.idxmax()} carried {share12.max():.0f}% of the {br['Passenger Number'].sum() / 1e6:.0f} million passengers in the last 12 months. "
    f"The highest load factor is {max(lf12, key=lf12.get)} ({max(lf12.values()):.0f}%), the lowest {min(lf12, key=lf12.get)} ({min(lf12.values()):.0f}%).", 520)

# 2.2 TREEMAP: the same hierarchy, area = passengers, colour = performance index
ids, labels, parents, values, cols = hierarchy(lambda n: float(PIL0["Index"].mean()) if n == "root" else float(PIL0.loc[n.split("/")[0], "Index"]))
fig = go.Figure(go.Treemap(ids=ids, labels=labels, parents=parents, values=values, branchvalues="total", marker=dict(colors=cols, colorscale="Viridis", cmin=0, cmax=100,
        colorbar=dict(title=dict(text="Index"))), textinfo="label+percent root", hovertemplate="%{label}<br>%{value:.2f} million passengers<extra></extra>"))
ix_top = PIL0["Index"].idxmax()
put("treemap", 2, "Is size the same as performance?", "Area = passengers in the last 12 months; colour = performance index (yellow = strong).", fig,
    f"{share12.idxmax()} fills {share12.max():.0f}% of the area at an index of {PIL0.loc[share12.idxmax(), 'Index']:.0f}, while {ix_top} (index {PIL0.loc[ix_top, 'Index']:.0f}) covers {share12.get(ix_top, 0):.0f}%. "
    f"Size and performance are different questions: the index rewards efficiency and momentum, not volume.", 500)

# 2.3 SANKEY: 2019 brands -> groups -> outcome
b = sd[sd.Year == 2019].groupby(["Airline", "Group"])["Passenger Number"].sum().reset_index(); b = b[b["Passenger Number"] > 0]
b["src"] = np.where(b["Passenger Number"] < 0.3e6, "Other small brands", b.Airline)
L = b.groupby(["src", "Group"])["Passenger Number"].sum().reset_index()
gout = L.groupby("Group")["Passenger Number"].sum()
OUTC = {"Recovered and holding": "#009E73", "Recovered, then fell back": "#E69F00", "Never recovered (still flying)": "#D55E00",
        "Never recovered (stopped flying)": "#6A3D9A", "New entrant (no 2019 level)": "#7F7F7F", "Not tested (partial or tiny 2019 base)": "#BBBBBB"}
rgba = lambda h, a: f"rgba({int(h[1:3], 16)},{int(h[3:5], 16)},{int(h[5:7], 16)},{a})"
nodes = [("b", s) for s in L.src.unique()] + [("g", g) for g in gout.index] + [("o", o) for o in sorted({STATUS[g] for g in gout.index})]
nid = {n: i for i, n in enumerate(nodes)}
src = [nid[("b", r.src)] for r in L.itertuples()] + [nid[("g", g)] for g in gout.index]
tgt = [nid[("g", r.Group)] for r in L.itertuples()] + [nid[("o", STATUS[g])] for g in gout.index]
val = [v / 1e6 for v in L["Passenger Number"]] + [v / 1e6 for v in gout.values]
lcol = ["rgba(150,150,150,.35)"] * len(L) + [rgba(OUTC[STATUS[g]], .45) for g in gout.index]
ncol = ["#bbbbbb" if t == "b" else (gc(n) if t == "g" else OUTC[n]) for t, n in nodes]
fig = go.Figure(go.Sankey(arrangement="snap", node=dict(label=[n for _, n in nodes], color=ncol, pad=16, thickness=16),
                          link=dict(source=src, target=tgt, value=val, color=lcol, hovertemplate="%{source.label} to %{target.label}: %{value:.1f} million<extra></extra>")))
fig.update_layout(margin=dict(l=20, r=20, t=20, b=20))
oc = pd.Series({o: gout[[g for g in gout.index if STATUS[g] == o]].sum() / 1e6 for o in OUTC if any(STATUS[g] == o for g in gout.index)})
T19 = gout.sum() / 1e6
put("sankey", 2, "Where did the 2019 passenger volume end up?", "Left = 2019 brands (width = 2019 passengers), middle = airline group, right = what happened to that group since.", fig,
    f"Of {T19:.0f} million passengers in 2019, {oc.get('Recovered and holding', 0) / T19 * 100:.0f}% flew with groups that are back above 2019 today, "
    f"{oc.get('Never recovered (stopped flying)', 0) / T19 * 100:.0f}% with groups that stopped flying without recovering, and {oc.get('Never recovered (still flying)', 0) / T19 * 100:.0f}% with groups still flying below 2019.", 540)

# 2.4 WATERFALL: how the market's monthly passengers changed from 2019 to now, group by group
avg19, avg12 = mk.loc[b19i].mean() / 1e6, mk.loc[W12].mean() / 1e6
dlt = (P["pax"].loc[W12].fillna(0).mean() - P["pax"].loc[b19i].fillna(0).mean()) / 1e6     # an airline absent in a period counts as zero
big = dlt[dlt.abs() >= 0.05].sort_values(ascending=False); rest = dlt.sum() - big.sum()
assert abs(avg19 + dlt.sum() - avg12) < 1e-6, "waterfall steps must add up to the end total"
xs = ["2019 average"] + big.index.tolist() + ["All other groups", "Last 12 months"]
fig = go.Figure(go.Waterfall(x=xs, measure=["absolute"] + ["relative"] * (len(big) + 1) + ["total"], y=[avg19] + big.tolist() + [rest, 0],
        text=[f"{avg19:.1f}"] + [f"{v:+.2f}" for v in big] + [f"{rest:+.2f}", f"{avg12:.1f}"], textposition="outside",
        increasing=dict(marker=dict(color="#009E73")), decreasing=dict(marker=dict(color="#D55E00")), totals=dict(marker=dict(color="#555555")), connector=dict(line=dict(color="#999"))))
fig.update_yaxes(title="Passengers per month (millions)", range=[0, max(avg19, avg12) * 1.3]); fig.update_xaxes(tickangle=-25)
gone = [g for g in big.index if g not in ORDER and big[g] < 0]
put("waterfall", 2, "Who added and who lost passengers?", "Start = the average month of 2019. Green steps add passengers, orange steps lose them; the dark bar is the average month of the last 12 months.", fig,
    f"The market grew from {avg19:.1f} to {avg12:.1f} million passengers a month ({(avg12 / avg19 - 1) * 100:+.0f}%). {big.index[0]} added {big.iloc[0]:+.2f} million a month and "
    f"{big.index[-1]} changed by {big.iloc[-1]:+.2f} million; airlines that stopped flying ({', '.join(gone) if gone else 'none'}) took {sum(big[g] for g in gone):+.2f} million a month with them.", 480)

# 2.5 TERNARY: the market split between IndiGo, Air India Group and everyone else, month by month
tm = P["pax"].loc["2019-01-01":]; tt = tm.sum(axis=1)
a, c2 = tm["IndiGo"] / tt * 100, tm["Air India Group"] / tt * 100
tmask = tt > 0
fig = go.Figure(go.Scatterternary(a=a[tmask], b=c2[tmask], c=(100 - a - c2)[tmask], mode="lines+markers", line=dict(color="#bbb", width=1),
        marker=dict(size=7, color=np.arange(tmask.sum()), colorscale="Viridis", showscale=True, colorbar=dict(title=dict(text="Months since<br>Jan 2019"))),
        text=[f"{d:%b %Y}" for d in tm.index[tmask]], hovertemplate="%{text}<br>IndiGo %{a:.0f}%<br>Air India Group %{b:.0f}%<br>Others %{c:.0f}%<extra></extra>"))
amin, bmin = max(0, np.floor(a[tmask].min() / 10) * 10 - 10), max(0, np.floor(c2[tmask].min() / 10) * 10 - 5)     # zoom in on the part of the triangle the data uses
cmin_ = max(0, min(np.floor((100 - a - c2)[tmask].min() / 10) * 10 - 5, 95 - amin - bmin))
fig.update_layout(ternary=dict(sum=100, aaxis=dict(title=dict(text="IndiGo %"), min=amin), baxis=dict(title=dict(text="Air India Group %"), min=bmin), caxis=dict(title=dict(text="All others %"), min=cmin_)))
put("ternary", 2, "How the three-way split moved", "Each dot is one month; a dot close to a corner means that party dominates. Dark = early 2019, yellow = recent.", fig,
    f"IndiGo's share rose from {a['2019'].mean():.0f}% in 2019 to {a.iloc[-12:].mean():.0f}% over the last 12 months and Air India Group's moved from {c2['2019'].mean():.0f}% to {c2.iloc[-12:].mean():.0f}%. "
    f"Everyone else fell from {(100 - a - c2)['2019'].mean():.0f}% to {(100 - a - c2).iloc[-12:].mean():.0f}%.", 500)

# 2.6 MARIMEKKO: width = share of passengers, height = load factor
mm = share12[share12 >= 0.3].sort_values(ascending=False); wd = mm / mm.sum() * 100; start = wd.cumsum() - wd
fig = go.Figure(go.Bar(x=(start + wd / 2).tolist(), y=[lf12[g] for g in wd.index], width=(wd * 0.97).tolist(), marker=dict(color=[gc(g) for g in wd.index], line=dict(color="white", width=1)),
        text=[f"{g}<br>{lf12[g]:.0f}%" if w > 8 else "" for g, w in wd.items()], textposition="inside", insidetextanchor="start", textfont=dict(color="white"),
        hovertext=[f"{g}: {w:.1f}% of passengers, load factor {lf12[g]:.1f}%" for g, w in wd.items()], hoverinfo="text"))
for g, w in wd.items():
    if 2.5 < w <= 8: fig.add_annotation(x=start[g] + w / 2, y=lf12[g], text=f"{g} {lf12[g]:.0f}%", textangle=-90, yanchor="bottom", showarrow=False, font=dict(size=10))
fig.update_xaxes(title="Cumulative share of passengers (%): bar width = the airline's share", range=[0, 100]); fig.update_yaxes(title="Load factor (% of seat-km filled)", range=[0, 135])
wavg = sum(lf12[g] * w for g, w in wd.items()) / 100
put("marimekko", 2, "Big and full at the same time?", "Width = share of passengers (last 12 months), height = load factor. The thinnest slivers on the right are labelled on hover.", fig,
    f"Weighted by passengers, the airlines shown fill {wavg:.0f}% of their seat-kilometres. {wd.index[0]} takes {wd.iloc[0]:.0f}% of the width at {lf12[wd.index[0]]:.0f}%, "
    f"while {min(wd.index, key=lambda g: lf12[g])} is the emptiest at {min(lf12[g] for g in wd.index):.0f}%.", 460)

# 2.7 FUNNEL: how many of the airlines that flew in 2019 came through
st = {g: STATUS[g] for g in BASE}
n0 = len(BASE); n1 = sum(1 for g in BASE if "still flying" in st[g] or st[g].startswith("Recovered")); n2 = sum(1 for g in BASE if st[g].startswith("Recovered")); n3 = sum(1 for g in BASE if st[g] == "Recovered and holding")
fig = go.Figure(go.Funnel(y=["Flew all of 2019 (and big enough to compare)", "Still flying today", "Reached 2019 passenger level again", "Still above 2019 level today"], x=[n0, n1, n2, n3],
        textinfo="value+percent initial", marker=dict(color=["#999999", "#56B4E9", "#E69F00", "#009E73"])))
fig.update_layout(margin=dict(l=280, r=30, t=30, b=40))
put("funnel", 2, "How many airlines made it through?", "Each stage keeps only the airlines that passed the previous test.", fig,
    f"Of {n0} airlines that flew all of 2019, {n1} are still flying, {n2} got back to their 2019 level and {n3} are above it now, "
    f"a {n3 / n0 * 100:.0f}% rate of staying above the 2019 level.", 400)


## 8. Section 3: time patterns (4 charts)

In [8]:
# ---------------- SECTION 3: TIME PATTERNS ----------------
MON = [calendar.month_abbr[m] for m in range(1, 13)]
# 3.1 POLAR BAR (wind rose): passengers by calendar month, 2019 vs the last 12 months
by19 = pd.Series(mk.loc[b19i].values / 1e6, index=b19i.month).sort_index(); by12 = pd.Series(mk.loc[W12].values / 1e6, index=W12.month).sort_index()
fig = go.Figure([go.Barpolar(r=by12, theta=MON, name=f"{W12[0]:%b %Y} to {W12[-1]:%b %Y}", marker=dict(color="#0072B2"), opacity=.55),
                 go.Barpolar(r=by19, theta=MON, name="2019", marker=dict(color="#333333", line=dict(color="white", width=1)), opacity=.8, width=0.45)])
fig.update_layout(polar=dict(angularaxis=dict(direction="clockwise", rotation=90), radialaxis=dict(ticksuffix=" M", angle=75, tickfont=dict(size=10)), barmode="overlay"))   # polar charts need polar.barmode; the default stacks
pk, tr_ = by12.idxmax(), by12.idxmin(); gap = (by12 / by19 - 1) * 100
put("polarbar", 3, "Which months are busy?", "Each wedge is a calendar month; length = million passengers. Narrow dark wedges = 2019, wide blue wedges = the last 12 months.", fig,
    f"The busiest month in the last 12 months is {MON[pk - 1]} ({by12.max():.1f} million) and the quietest {MON[tr_ - 1]} ({by12.min():.1f} million), a gap of {(by12.max() / by12.min() - 1) * 100:.0f}%. "
    + (f"Every month beats its 2019 level, by {gap.min():.0f}% to {gap.max():.0f}%." if gap.min() > 0 else f"{int((gap > 0).sum())} of 12 months beat 2019; {MON[gap.idxmin() - 1]} is {gap.min():.0f}% below."), 500)

# 3.2 3D SURFACE: the market as a landscape of year x month
full = pd.date_range("2017-01-01", f"{LAST.year}-12-01", freq="MS"); mz = mk.reindex(full) / 1e6
Z = pd.DataFrame({"y": full.year, "m": full.month, "v": mz.values}).pivot(index="y", columns="m", values="v")
fig = go.Figure(go.Surface(z=Z.values, x=MON, y=Z.index.tolist(), colorscale="Viridis", colorbar=dict(title=dict(text="Million")), connectgaps=False))
fig.update_layout(scene=dict(xaxis=dict(title=dict(text="Month")), yaxis=dict(title=dict(text="Year")), zaxis=dict(title=dict(text="Passengers (M)")), camera=dict(eye=dict(x=1.6, y=-1.6, z=.9))),
                  margin=dict(l=0, r=0, t=10, b=0))
put("surface", 3, "The market as a landscape", "Drag to rotate. Height = passengers in that month; the canyon is the lockdown, the high ground at the back is the post-COVID market.", fig,
    f"The lowest point is {mk.loc['2020-01-01':].idxmin():%B %Y} at {mk.loc['2020-01-01':].min() / 1e6:.1f} million; the highest is {mk.idxmax():%B %Y} at {mk.max() / 1e6:.1f} million, "
    f"{(mk.max() / mk.loc[b19i].max() - 1) * 100:.0f}% above the best month of 2019.", 540)

# 3.3 CANDLESTICK: quarterly range of monthly passengers for IndiGo
s = P["pax"]["IndiGo"]["2022-01-01":] / 1e6; q = s.groupby(s.index.to_period("Q"))
oh = pd.DataFrame({"o": q.first(), "h": q.max(), "l": q.min(), "c": q.last()}); xq = oh.index.to_timestamp()
fig = go.Figure(go.Candlestick(x=xq, open=oh.o, high=oh.h, low=oh.l, close=oh.c, increasing=dict(line=dict(color="#009E73"), fillcolor="#009E73"),
                               decreasing=dict(line=dict(color="#D55E00"), fillcolor="#FFFFFF"), name="IndiGo"))
fig.update_layout(xaxis_rangeslider_visible=False); fig.update_yaxes(title="IndiGo passengers per month (millions)"); fig.update_xaxes(title="Quarter")
rng_ = (oh.h - oh.l) / oh.o * 100
put("candle", 3, "How much does a quarter swing?", "One candle per quarter: body = first to last month, wick = highest to lowest month. Filled green = ended higher; hollow orange = ended lower.", fig,
    f"IndiGo's passengers ended higher than they began in {int((oh.c > oh.o).sum())} of {len(oh)} quarters since 2022. "
    f"The widest quarterly swing was {rng_.max():.0f}% ({xq[rng_.argmax()].to_period('Q')}), the narrowest {rng_.min():.0f}%.", 450)

# 3.4 FAN CHART: a 6-month projection of market passengers, with the error the rule actually made on past months
fc = forecast(mk, LAST); fx = pd.date_range(LAST + pd.DateOffset(months=1), periods=6, freq="MS")
band = fc.values * (MAPE.values / 100)
hist_ = mk.loc[LAST - pd.DateOffset(months=35):] / 1e6
px_ = [LAST] + list(fx); py = [mk.loc[LAST] / 1e6] + list(fc.values / 1e6)
up_ = [mk.loc[LAST] / 1e6] + list((fc.values + band) / 1e6); lo_ = [mk.loc[LAST] / 1e6] + list((fc.values - band) / 1e6)
fig = go.Figure([go.Scatter(x=px_ + px_[::-1], y=up_ + lo_[::-1], fill="toself", mode="lines", fillcolor="rgba(0,114,178,.18)", line=dict(width=0), name="Typical error band", hoverinfo="skip"),
                 go.Scatter(x=hist_.index, y=hist_, mode="lines", name="Actual", line=dict(color="#333", width=2.5)),
                 go.Scatter(x=px_, y=py, mode="lines+markers", name="Projection", line=dict(color="#0072B2", width=2.5, dash="dash"), marker=dict(size=8))])
fig.update_yaxes(title="Passengers per month (millions)", rangemode="tozero"); fig.update_xaxes(title="Month")
put("fan", 3, "What might the next six months look like?", "Projection = the same month last year x the latest 12-month growth rate. The band is the average error this rule made when tested on past months.", fig,
    f"The rule projects {fc.iloc[0] / 1e6:.1f} million passengers in {fx[0]:%B %Y} and {fc.iloc[-1] / 1e6:.1f} million in {fx[-1]:%B %Y}. "
    f"Tested on {len(errs[1])} past months, its average error was {MAPE[1]:.1f}% one month ahead and {MAPE[6]:.1f}% six months ahead. It is a rule of thumb, not a forecast model.", 450)


## 9. Section 4: distributions (4 charts)

In [9]:
# ---------------- SECTION 4: DISTRIBUTIONS ----------------
LFm = P["rpk"] / P["ask"] * 100                                   # monthly load factor
YOY = (P["pax"] / P["pax"].shift(12) - 1) * 100                   # monthly year-on-year growth
since = months >= "2023-06-01"
# 4.1 BOX PLOT: monthly passengers vs the same month of 2019, airlines with a 2019 level
act_base = [g for g in ORDER if g in BASE]
fig = go.Figure([go.Box(y=(RATIO.loc[since, g] * 100).dropna(), name=g, marker=dict(color=gc(g), symbol=gs(g)), boxpoints="all", jitter=.4, pointpos=-1.7, boxmean=True) for g in act_base])
fig.add_hline(y=100, line=dict(dash="dash", color="#222"), annotation_text="2019 level", annotation_position="top right")
fig.update_yaxes(title="% of the same month in 2019"); fig.update_layout(showlegend=False)
med = {g: (RATIO.loc[since, g] * 100).median() for g in act_base}
above = [g for g in act_base if (RATIO.loc[since, g] * 100).quantile(.25) > 100]
put("box", 4, "How wide is the monthly spread against 2019?", "Box = middle half of the months, solid line = median, dashed line = mean, dots = individual months since mid-2023.", fig,
    f"{max(med, key=med.get)} has a median of {max(med.values()):.0f}% of 2019 and {min(med, key=med.get)} only {min(med.values()):.0f}%. "
    + (f"The boxes of {' and '.join(above)} sit entirely above the 2019 line." if above else "No airline's box sits entirely above the 2019 line."), 460)

# 4.2 VIOLIN: shape of each airline's monthly load factor
fig = go.Figure([go.Violin(y=LFm.loc[since, g].dropna(), name=g, line=dict(color=gc(g)), fillcolor=gc(g), opacity=.55, box_visible=True, meanline_visible=True,
                           points="all", pointpos=0, jitter=.3, marker=dict(size=3)) for g in ORDER])
fig.update_yaxes(title="Monthly load factor (% of seat-km filled)"); fig.update_layout(showlegend=False)
sd_lf = LFm.loc[since, ORDER].std().sort_values()
put("violin", 4, "How steady is the load factor?", "The width of each shape shows how often the airline had that load factor. A short, fat shape = a steady airline.", fig,
    f"{sd_lf.index[0]} is the steadiest (standard deviation {sd_lf.iloc[0]:.1f} points) and {sd_lf.index[-1]} the most erratic ({sd_lf.iloc[-1]:.1f} points). "
    f"Median load factors range from {LFm.loc[since, ORDER].median().min():.0f}% to {LFm.loc[since, ORDER].median().max():.0f}%.", 480)

# 4.3 HISTOGRAM with a rug: monthly year-on-year growth of all scored airlines
yy = YOY.loc[since, ORDER].stack().clip(-100, 150)
fig = make_subplots(rows=2, cols=1, row_heights=[.82, .18], shared_xaxes=True, vertical_spacing=.03)
fig.add_trace(go.Histogram(x=yy, nbinsx=40, marker=dict(color="#0072B2", line=dict(color="white", width=1)), name="Airline-months", showlegend=False), row=1, col=1)
fig.add_trace(go.Scatter(x=yy, y=np.zeros(len(yy)), mode="markers", marker=dict(symbol="line-ns-open", size=14, color="#555"), showlegend=False, hoverinfo="x"), row=2, col=1)
fig.add_vline(x=0, line=dict(dash="dash", color="#222"))
fig.update_yaxes(title="Number of airline-months", row=1, col=1); fig.update_yaxes(visible=False, row=2, col=1)
fig.update_xaxes(title="Year-on-year growth in passengers (%, capped at -100 and +150)", row=2, col=1)
put("hist", 4, "How often do airlines grow?", "Each bar counts airline-months with that year-on-year growth. The tick marks underneath are the individual observations.", fig,
    f"{(yy > 0).mean() * 100:.0f}% of the {len(yy)} airline-months grew against the same month a year earlier. "
    f"The median is {yy.median():+.1f}% and the middle half runs from {yy.quantile(.25):+.0f}% to {yy.quantile(.75):+.0f}%.", 440)

# 4.4 DENSITY CONTOUR: load factor against growth for every airline-month
dd = pd.DataFrame({"lf": LFm.loc[since, ORDER].stack(), "yoy": YOY.loc[since, ORDER].stack().clip(-100, 150)}).dropna()
fig = go.Figure([go.Histogram2dContour(x=dd.lf, y=dd.yoy, colorscale=[[0, "#ffffff"], [0.12, "#deebf7"], [0.5, "#4292c6"], [1, "#08306b"]], ncontours=12, showscale=True,
                                       colorbar=dict(title=dict(text="Months")), contours=dict(coloring="fill"), line=dict(width=0.5, color="#ffffff")),
                 go.Scatter(x=dd.lf, y=dd.yoy, mode="markers", marker=dict(size=3, color="rgba(40,40,40,.55)"), hoverinfo="skip", showlegend=False)])
fig.update_xaxes(title="Monthly load factor (%)"); fig.update_yaxes(title="Year-on-year growth in passengers (%)")
put("contour", 4, "Do fuller planes mean faster growth?", "Contours show where airline-months cluster (darker = more months); grey dots are the individual months.", fig,
    f"The densest cluster sits near a load factor of {dd.lf.median():.0f}% and growth of {dd.yoy.median():+.0f}%. The rank correlation between load factor and growth is {spearman(dd.lf, dd.yoy):+.2f}, "
    f"so fuller planes alone do not signal faster growth.", 460)


## 10. Section 5: relationships and prediction (6 charts)

In [10]:
# ---------------- SECTION 5: RELATIONSHIPS AND PREDICTION ----------------
# 5.1 SCATTER MATRIX: the six common metrics, every airline at every month since mid-2023
lab_s = {"lf": "Load factor %", "ppd": "Pax per dep.", "prod": "Productivity", "gro": "Growth %", "shm": "Share mom. pp", "vol": "Volatility"}
CLIP = {"gro": (-100, 150), "vol": (0, 100)}      # display only: launch-year values (volatility near 600) would squash every other point
cv = lambda k, s: s.clip(*CLIP[k]) if k in CLIP else s
fig = go.Figure([go.Splom(dimensions=[dict(label=lab_s[k], values=cv(k, OBS.loc[OBS.g == g, k])) for k in COMMON], name=g,
                          marker=dict(color=gc(g), symbol=gs(g), size=5, opacity=.8), diagonal_visible=False, showupperhalf=False) for g in ORDER if (OBS.g == g).any()])
fig.update_layout(dragmode="select")
rho = OBS[COMMON].corr(method="spearman").where(lambda d: np.tril(np.ones(d.shape), -1).astype(bool)).stack()
hi_, lo_ = rho.abs().idxmax(), rho.abs().idxmin()
put("splom", 5, "Which metrics move together?", "Every dot is one airline in one month. Each panel plots two metrics against each other; colours and symbols are airlines. Growth is clipped at 150% and volatility at 100 for display.", fig,
    f"The strongest pairing is {MET[hi_[0]][0].lower()} with {MET[hi_[1]][0].lower()} (rank correlation {rho[hi_]:+.2f}). "
    f"The weakest is {MET[lo_[0]][0].lower()} with {MET[lo_[1]][0].lower()} ({rho[lo_]:+.2f}), so those two carry separate information.", 760)

# 5.2 DENDROGRAM: which airlines are alike? (Ward clustering on the six common metric scores)
X = S0.loc[ORDER, COMMON].fillna(50)
ZL = linkage(X.values, "ward"); THR = (ZL[-3, 2] + ZL[-2, 2]) / 2          # cut height that leaves exactly three groups, so colours match the caption
fig = ff.create_dendrogram(X.values, labels=ORDER, orientation="left", linkagefun=lambda x: linkage(x, "ward"), color_threshold=THR,
                           colorscale=["#0072B2", "#D55E00", "#009E73", "#CC79A7", "#E69F00", "#56B4E9", "#A0522D", "#333333"])
fig.update_xaxes(title="Distance between airlines (lower = more alike)"); fig.add_vline(x=THR, line=dict(dash="dot", color="#888"))
cl = fcluster(linkage(X.values, "ward"), t=3, criterion="maxclust")
groups_ = [[g for g, k in zip(ORDER, cl) if k == c] for c in sorted(set(cl))]
put("dendro", 5, "Which airlines are alike?", "Airlines joined by a short branch have similar metric profiles. The dotted line cuts the tree into three peer groups, one colour each.", fig,
    "Three peer groups emerge: " + "; ".join(", ".join(v) for v in groups_) + ". Comparing an airline with its own peer group is fairer than comparing it with the whole market.", 430)

# 5.3 BUBBLE CHART: growth vs load factor, size = passengers, colour = index
xb, yb = M0.loc[ORDER, "gro"], M0.loc[ORDER, "lf"]; sz = np.array([P["pax"].loc[W12, g].mean() / 1e6 for g in ORDER])
fig = go.Figure(go.Scatter(x=xb, y=yb, mode="markers+text", text=ORDER, textposition=["bottom center" if g == "Air India Group" else "top center" for g in ORDER], marker=dict(size=np.sqrt(sz) * 22 + 12, sizemode="diameter",
        color=PIL0.loc[ORDER, "Index"], colorscale="Viridis", cmin=0, cmax=100, showscale=True, colorbar=dict(title=dict(text="Index")), line=dict(color="#222", width=1), opacity=.85),
        hovertemplate="%{text}<br>growth %{x:.0f}%<br>load factor %{y:.1f}%<extra></extra>"))
fig.add_vline(x=float(xb.median()), line=dict(dash="dot", color="#888")); fig.add_hline(y=float(yb.median()), line=dict(dash="dot", color="#888"))
fig.update_xaxes(title="Growth momentum: passengers vs previous 12 months (%)"); fig.update_yaxes(title="Load factor, last 12 months (%)", range=[55, 100])
q_ = [g for g in ORDER if xb[g] > xb.median() and yb[g] > yb.median()]
put("bubble", 5, "Growing and full?", "Right = growing faster, up = fuller planes. Bubble size = passengers per month; colour = performance index. Dotted lines are the peer medians.", fig,
    (f"{' and '.join(q_)} {'is' if len(q_) == 1 else 'are'} above the median on both growth and load factor. " if q_ else "No airline is above the median on both growth and load factor. ")
    + f"{xb.idxmax()} grows fastest ({xb.max():+.0f}%) from a small base; {xb.idxmin()} is shrinking fastest ({xb.min():+.0f}%).", 500)

# 5.4 3D SCATTER: load factor x passengers per departure x network productivity
fig = go.Figure(go.Scatter3d(x=M0.loc[ORDER, "lf"], y=M0.loc[ORDER, "ppd"], z=M0.loc[ORDER, "prod"], mode="markers+text", text=ORDER, textposition="top center",
        marker=dict(size=9, color=PIL0.loc[ORDER, "Index"], colorscale="Viridis", cmin=0, cmax=100, showscale=True, colorbar=dict(title=dict(text="Index")), line=dict(color="#222", width=1))))
fig.update_layout(scene=dict(xaxis=dict(title=dict(text="Load factor %")), yaxis=dict(title=dict(text="Passengers / departure")), zaxis=dict(title=dict(text="Pax-km per aircraft hour (000)"))),
                  margin=dict(l=0, r=0, t=10, b=0))
put("scatter3d", 5, "The operating-efficiency cube", "Drag to rotate. The three axes are the Operations pillar; colour is the overall index.", fig,
    f"{M0.loc[ORDER, 'prod'].idxmax()} leads on productivity ({M0['prod'].max():.0f} thousand passenger-km per aircraft hour) and {M0.loc[ORDER, 'ppd'].idxmax()} on passengers per departure ({M0['ppd'].max():.0f}). "
    f"{M0['prod'].idxmin()} sits at the opposite corner ({M0['prod'].min():.0f}): small aircraft on short sectors, which this pillar structurally penalises.", 560)

# 5.5 LOLLIPOP: which metrics actually line up with the next 6 months of growth?
lab = {**{k: MET[k][0] for k in MET}, **{p: p + " pillar" for p in PILLARS}, "Index": "Performance index"}
ps = PRED.sort_values()
xs, ys = [], []
for k, v in ps.items(): xs += [0, v, None]; ys += [lab[k], lab[k], None]
fig = go.Figure([go.Scatter(x=xs, y=ys, mode="lines", line=dict(color="#bbb", width=3), hoverinfo="skip", showlegend=False),
                 go.Scatter(x=ps.values, y=[lab[k] for k in ps.index], mode="markers+text", text=[f"{v:+.2f}" for v in ps.values],
                            textposition=["middle right" if v >= 0 else "middle left" for v in ps.values],
                            marker=dict(size=13, color=["#009E73" if v > .3 else ("#D55E00" if v < -.1 else "#999") for v in ps.values],
                                        symbol=["circle" if v > .3 else ("x" if v < -.1 else "circle-open") for v in ps.values], line=dict(color="#222", width=1)),
                            customdata=[int(NOBS[k]) for k in ps.index], hovertemplate="%{y}: %{x:+.2f} (n = %{customdata})<extra></extra>", showlegend=False)])
fig.add_vline(x=0, line=dict(color="#222", width=1)); fig.update_xaxes(title="Rank correlation with passenger growth over the next 6 months", range=[-.65, 1.05])
bestm = ps.drop(["Index"] + PILLARS).idxmax(); worstm = ps.drop(["Index"] + PILLARS).idxmin()
put("lollipop", 5, "Which metrics predict what happens next?", "Each stick shows how strongly a metric measured today lines up with growth over the next 6 months (+1 = perfectly, 0 = not at all). Signed so that positive means the metric's 'better' direction goes with faster growth. Green dot = useful, cross = misleading.", fig,
    f"{MET[bestm][0]} is the best predictor ({ps[bestm]:+.2f}) and {MET[worstm][0].lower()} the worst ({ps[worstm]:+.2f}). The combined index scores {ps['Index']:+.2f}: "
    f"it describes current performance rather than future growth. Based on {len(OBS)} airline-months with overlapping windows, so treat it as indicative.", 560)

# 5.6 RANGE DOT PLOT: the Go First backtest
labs = PILLARS + ["Index"]; peers = BT_P.drop(index="Go First")
xs, ys = [], []
for p in labs: xs += [peers[p].min(), peers[p].max(), None]; ys += [p, p, None]
fig = go.Figure([go.Scatter(x=xs, y=ys, mode="lines", line=dict(color="#ccc", width=12), name="Range of the other airlines", hoverinfo="skip"),
                 go.Scatter(x=[peers[p].median() for p in labs], y=labs, mode="markers", marker=dict(symbol="line-ns", size=18, color="#555", line=dict(width=3, color="#555")), name="Peer median"),
                 go.Scatter(x=[BT_P.loc["Go First", p] for p in labs], y=labs, mode="markers+text", text=[f"{BT_P.loc['Go First', p]:.0f}" for p in labs], textposition="top center",
                            marker=dict(size=15, color=gc("Go First"), symbol="diamond", line=dict(color="#222", width=1)), name="Go First")])
fig.update_xaxes(title=f"Score as of {BT_T:%B %Y}, using only data up to then", range=[0, 100]); fig.update_yaxes(autorange="reversed")
sp = BT_P["Index"].drop("Go First").sort_values()
put("backtest", 5, "Would the metrics have flagged Go First?", "Grey bar = the range of the other airlines' scores in the month Go First stopped flying; dark tick = their median; diamond = Go First.", fig,
    f"As of {BT_T:%B %Y}, Go First ranked {BT_rank['Go First']} of {len(BT_P)} on the index ({BT_P.loc['Go First', 'Index']:.0f}), with the weakest momentum pillar ({BT_P.loc['Go First', 'Momentum']:.0f}). "
    f"The margin was thin: {sp.index[0]} scored {sp.iloc[0]:.0f} and is still flying. Traffic data shows weak momentum but cannot see a funding crisis.", 430)


## 11. Conclusions

In [11]:
# CONCLUSIONS: every number below is computed from the data above
tot19, tot12m = mk.loc[b19i].mean() / 1e6, mk.loc[W12].mean() / 1e6
hhi = lambda sh: float((sh ** 2).sum() * 1e4)
sh19, sh12 = P["pax"].loc[b19i].sum() / P["pax"].loc[b19i].sum().sum(), P["pax"].loc[W12].sum() / P["pax"].loc[W12].sum().sum()
REC = pd.DataFrame({g: dict(when=ROLL.loc["2020-07-01":, g].dropna()[lambda s: s >= 1].index.min(), now=RATIO[g].iloc[-3:].mean() * 100) for g in BASE}).T
firstg = REC.dropna(subset=["when"]).sort_values("when")
mkt_ratio = (mk / pd.Series(mk.loc[b19i].values, index=b19i.month).reindex(mk.index.month).values).rolling(3).mean()
mkt_when = mkt_ratio.loc["2020-07-01":][lambda s: s >= 1].index.min()
never = REC[REC.when.isna()]
bestp, worstp = PRED.drop(["Index"] + PILLARS).idxmax(), PRED.drop(["Index"] + PILLARS).idxmin()
second = PRED.drop(["Index"] + PILLARS + [bestp]).idxmax()
CONCL = [
  ("The market is back; many airlines are not.",
   f"Passengers per month averaged {tot12m:.1f} million over the last 12 months against {tot19:.1f} million in 2019 ({(tot12m / tot19 - 1) * 100:+.0f}%), and the 3-month average first matched 2019 in {mkt_when:%B %Y}. "
   f"Of the {len(BASE)} airlines that flew all of 2019, {len(firstg)} regained their 2019 level ("
   + "; ".join(f"{g} in {d:%B %Y}" for g, d in firstg["when"].items())
   + f"), but only {sum(1 for g in BASE if STATUS[g] == 'Recovered and holding')} are still above it. {', '.join(never.index)} never got back."),
  ("Concentration has risen sharply.",
   f"IndiGo's share went from {sh19['IndiGo'] * 100:.0f}% to {sh12['IndiGo'] * 100:.0f}% and the market concentration index (HHI) from {hhi(sh19):,.0f} to {hhi(sh12):,.0f} (above 2,500 counts as highly concentrated). "
   f"Air India Group moved from {sh19['Air India Group'] * 100:.0f}% to {sh12['Air India Group'] * 100:.0f}% and SpiceJet from {sh19['SpiceJet'] * 100:.0f}% to {sh12['SpiceJet'] * 100:.0f}%."),
  ("The index separates the airlines clearly, and the order is robust.",
   f"Among airlines scored on all three pillars, {FULL.index[0]} leads ({FULL.Index.iloc[0]:.0f}) ahead of {FULL.index[1]} ({FULL.Index.iloc[1]:.0f}), and {FULL.index[-1]} is last ({FULL.Index.iloc[-1]:.0f}). "
   f"{top} scores highest overall ({PIL0.loc[top, 'Index']:.0f}) on two pillars, as it has no 2019 level. No airline's rank moves by more than {int((RKW.max(axis=1) - RKW.min(axis=1)).max())} places across four different weightings."),
  ("Growth and consistency predict; efficiency does not.",
   f"Of the ten metrics, {MET[bestp][0].lower()} lines up best with the next six months of growth ({PRED[bestp]:+.2f}, partly because growth persists), followed by {MET[second][0].lower()} ({PRED[second]:+.2f}). "
   f"The Operations pillar scores {PRED['Operations']:+.2f} and {MET[worstp][0].lower()} {PRED[worstp]:+.2f}. The combined index scores {PRED['Index']:+.2f}: it describes how an airline is doing today, while growth says where it is heading."),
  ("A backtest shows both the use and the limit.",
   f"Using only data up to {BT_T:%B %Y}, Go First ranked {BT_rank['Go First']} of {len(BT_P)} (index {BT_P.loc['Go First', 'Index']:.0f}), dragged down by the weakest momentum pillar. "
   f"The gap to the next airline was small, and traffic data cannot see finances, so a score like this is a warning light, not a verdict."),
]
LIMITS = ["Scores are relative: 50 means the average of the airlines scored in that month, not an absolute standard.",
          f"Only {len(ORDER)} airlines are scored, so one airline can move the averages. The rank correlations rest on overlapping 12-month windows and are indicative.",
          "Operations metrics favour large jets on long sectors (passengers per departure, productivity) and rank regional turboprop operators lower regardless of management quality.",
          "DGCA data is provisional, and merged airlines are combined into groups (Air India Group = Air India, Air India Express, Vistara, AirAsia India, AIX Connect).",
          f"The analysis stops at {LAST:%B %Y}, the last month in which the airline rows add up to DGCA's own total within 2%."]
for t, b_ in CONCL: print("-", t, b_, "\n")


- The market is back; many airlines are not. Passengers per month averaged 13.9 million over the last 12 months against 12.0 million in 2019 (+16%), and the 3-month average first matched 2019 in February 2023. Of the 6 airlines that flew all of 2019, 3 regained their 2019 level (IndiGo in May 2022; Air India Group in December 2022; Alliance Air in December 2022), but only 2 are still above it. Go First, SpiceJet, TruJet never got back. 

- Concentration has risen sharply. IndiGo's share went from 47% to 64% and the market concentration index (HHI) from 3,097 to 4,851 (above 2,500 counts as highly concentrated). Air India Group moved from 23% to 26% and SpiceJet from 15% to 3%. 

- The index separates the airlines clearly, and the order is robust. Among airlines scored on all three pillars, IndiGo leads (66) ahead of Air India Group (58), and Alliance Air is last (23). Akasa Air scores highest overall (72) on two pillars, as it has no 2019 level. No airline's rank moves by more than 2 p

## 12. Build the HTML page

In [12]:
# ASSEMBLE ONE SELF-CONTAINED HTML PAGE (Plotly.js is embedded, so it opens offline in any browser)
esc = _html.escape
CFG = dict(displaylogo=False, responsive=True, modeBarButtonsToRemove=["lasso2d", "select2d"])
kpis = [(f"{tot12m / tot19 * 100:.0f}%", "of 2019 passengers", f"{tot12m:.1f} vs {tot19:.1f} million a month"),
        (FULL.index[0], "top airline on all three pillars", f"index {FULL.Index.iloc[0]:.0f} of 100"),
        (MET[bestp][0], "best predictor of next-6-month growth", f"rank correlation {PRED[bestp]:+.2f}"),
        (f"{BT_rank['Go First']} of {len(BT_P)}", "Go First's rank in its last month", f"index {BT_P.loc['Go First', 'Index']:.0f} vs best {BT_P['Index'].max():.0f}")]
kpi_html = "".join(f'<div class="kpi"><div class="v">{esc(a)}</div><div class="l">{esc(b_)}</div><div class="s">{esc(c)}</div></div>' for a, b_, c in kpis)
met_rows = "".join(f"<tr><td>{p}</td><td><b>{esc(MET[k][0])}</b></td><td>{esc(MET[k][1])}</td><td>{'higher' if MET[k][2] else 'lower'}</td><td>{esc(WHY[k])}</td></tr>"
                   for p in PILLARS for k in MET if MET[k][3] == p)
nav = "".join(f'<a href="#s{n}">{n}. {esc(t.split(":")[0])}</a>' for n, t in SEC.items()) + '<a href="#concl">Conclusions</a><a href="#metrics">The ten metrics</a>'
secs = ""
for n, t in SEC.items():
    cards = "".join(f'<section class="card" id="c-{k}"><h3>{esc(v["title"])}</h3><p class="how">{esc(v["how"])}</p>'
                    f'{v["fig"].to_html(full_html=False, include_plotlyjs=False, div_id="fig-" + k, config=CFG)}'
                    f'<p class="cap"><b>What it shows.</b> {esc(v["cap"])}</p></section>' for k, v in FIGS.items() if v["sec"] == n)
    secs += f'<h2 id="s{n}">{n}. {esc(t)}</h2>{cards}'
concl = "".join(f"<li><b>{esc(t)}</b> {esc(b_)}</li>" for t, b_ in CONCL)
page = f"""<!doctype html><html lang="en"><head><meta charset="utf-8"><meta name="viewport" content="width=device-width, initial-scale=1">
<title>Indian airlines after COVID: performance dashboard</title>
<style>
:root{{--ink:#222;--mut:#5f5f5f;--line:#e2e2e2;--bg:#f6f7f9;--acc:#0072B2}}
*{{box-sizing:border-box}} body{{margin:0;font-family:system-ui,'Segoe UI',Arial,sans-serif;color:var(--ink);background:var(--bg);line-height:1.5}}
body.grey{{filter:grayscale(1)}}
header{{background:#fff;border-bottom:1px solid var(--line);padding:22px 28px}} h1{{margin:0 0 4px;font-size:26px}} .sub{{color:var(--mut);margin:0;max-width:920px}}
nav{{position:sticky;top:0;z-index:5;background:#fff;border-bottom:1px solid var(--line);padding:8px 28px;display:flex;gap:6px;flex-wrap:wrap;align-items:center}}
nav a,nav button{{color:var(--ink);text-decoration:none;font:inherit;font-size:14px;padding:5px 11px;border-radius:14px;background:var(--bg);border:0;cursor:pointer}}
nav a:hover,nav button:hover{{background:var(--acc);color:#fff}} nav button{{margin-left:auto;border:1px solid var(--line)}}
main{{max-width:1180px;margin:0 auto;padding:18px 16px 60px}} h2{{margin:34px 0 8px;font-size:21px;scroll-margin-top:60px}}
.kpis{{display:grid;grid-template-columns:repeat(auto-fit,minmax(210px,1fr));gap:12px;margin:18px 0}}
.kpi{{background:#fff;border:1px solid var(--line);border-radius:10px;padding:14px 16px}} .kpi .v{{font-size:23px;font-weight:700}} .kpi .l{{font-size:13px}} .kpi .s{{font-size:12px;color:var(--mut)}}
.card{{background:#fff;border:1px solid var(--line);border-radius:10px;padding:14px 16px 12px;margin:14px 0;scroll-margin-top:60px;overflow-x:auto;overflow-y:hidden}}
main{{overflow-x:hidden}} .card h3{{margin:0 0 2px;font-size:17px}}
.how{{margin:0 0 6px;color:var(--mut);font-size:13px}} .cap{{margin:8px 0 0;padding:9px 12px;background:#f3f6f9;border-left:4px solid var(--acc);font-size:14px}}
.tbl{{overflow-x:auto}} table{{border-collapse:collapse;width:100%;font-size:13px;background:#fff}} th,td{{border:1px solid var(--line);padding:6px 9px;text-align:left;vertical-align:top}} th{{background:#222;color:#fff}}
.box{{background:#fff;border:1px solid var(--line);border-radius:10px;padding:6px 20px 14px}} li{{margin:8px 0}} .small{{font-size:13px;color:var(--mut)}}
</style><script>{get_plotlyjs()}</script></head><body>
<header><h1>Indian airlines after COVID: performance dashboard</h1>
<p class="sub">Ten metrics, a 0 to 100 performance index for every airline, and {len(FIGS)} charts, each a different chart form. Scheduled domestic flights, January 2017 to {LAST:%B %Y}. Source: DGCA monthly traffic statistics.</p></header>
<nav>{nav}<button onclick="document.body.classList.toggle('grey')" title="Shows the whole page in greyscale, to check the charts still read without colour">Greyscale check</button></nav><main>
<div class="kpis">{kpi_html}</div>
<div class="box"><h3>How to read this page</h3><p class="small">Each airline is measured on ten metrics over the 12 months ending {LAST:%B %Y}. A metric score is 50 for the peer average and moves 25 points per standard deviation, capped to 0 to 100. The metrics form three pillars (Recovery, Operations, Momentum) and the index is the average of the pillars an airline has: airlines that started after 2019 have no Recovery pillar. An airline is scored if it averaged at least {MIN_PAX:,} passengers a month. Charts are interactive: hover for values, click legend names, drag to zoom or rotate.</p></div>
{secs}
<h2 id="concl">Conclusions</h2><div class="box"><ol>{concl}</ol><h3>Limitations</h3><ul>{"".join(f"<li class='small'>{esc(x)}</li>" for x in LIMITS)}</ul></div>
<h2 id="metrics">The ten metrics</h2><div class="tbl"><table><tr><th>Pillar</th><th>Metric</th><th>Unit</th><th>Better when</th><th>Why it matters</th></tr>{met_rows}</table></div>
<p class="small">Data: DGCA (Government of India) monthly traffic and operating statistics, parsed by the team and cross-checked against Vonter/india-aviation-traffic (ODbL according to the repository README). Charts: Plotly. Page generated {TODAY}.</p>
</main></body></html>"""
OUT_FILE = OUT / f"dgca_performance_dashboard_{TODAY}.html"
OUT_FILE.write_text(page, encoding="utf-8")
print(f"saved {OUT_FILE}: {OUT_FILE.stat().st_size / 1e6:.1f} MB | {len(FIGS)} charts | works offline")


saved /home/claude/proj/DGCA_aviation_analysis/outputs/dashboard/dgca_performance_dashboard_2026-11-04.html: 5.2 MB | 28 charts | works offline


## 13. The charts in the notebook

In [13]:
# SHOW EVERY CHART HERE TOO, with its caption (set SHOW_FIGURES = False in the first cell to skip)
if SHOW_FIGURES:
    for n, t in SEC.items():
        display(Markdown(f"## {n}. {t}"))
        for k, v in FIGS.items():
            if v["sec"] == n:
                display(Markdown(f"**{v['title']}** *{v['how']}*"))
                v["fig"].show(config=CFG)
                display(Markdown(f"**What it shows.** {v['cap']}"))
    display(Markdown("## Conclusions\n" + "\n".join(f"{i}. **{t}** {b_}" for i, (t, b_) in enumerate(CONCL, 1))))


## 1. Scoreboard: who performs best?

**How does each airline score today?** *Filled arc and number = performance index (0 to 100, 50 = peer average). The small arrow is the change since a year ago; no arrow = not scored a year ago.*

**What it shows.** Akasa Air has the highest index (72) and Alliance Air the lowest (23). Among the airlines scored on all three pillars, IndiGo leads (66); airlines without a 2019 level are scored on two pillars only.

**The full scorecard** *Coloured cells are scores (yellow = strong, purple = weak). Plain cells are the raw metric values in the units listed under 'The ten metrics'. n/a = no 2019 level to compare with.*

**What it shows.** Akasa Air scores 72 on a load factor of 92% and 173 passengers per departure. IndiGo follows at 66; Alliance Air is last at 23 with a load factor of 63%.

**What shape is each airline?** *Each spoke is one metric score (outer edge = best among peers). Click names in the legend to add or remove airlines.*

... (52 outputs left out)

**What it shows.** Growth momentum is the best predictor (+0.79) and load-factor change vs 2019 the worst (-0.36). The combined index scores +0.09: it describes current performance rather than future growth. Based on 190 airline-months with overlapping windows, so treat it as indicative.

**Would the metrics have flagged Go First?** *Grey bar = the range of the other airlines' scores in the month Go First stopped flying; dark tick = their median; diamond = Go First.*

**What it shows.** As of May 2023, Go First ranked 5 of 5 on the index (36), with the weakest momentum pillar (17). The margin was thin: SpiceJet scored 38 and is still flying. Traffic data shows weak momentum but cannot see a funding crisis.

## Conclusions
1. **The market is back; many airlines are not.** Passengers per month averaged 13.9 million over the last 12 months against 12.0 million in 2019 (+16%), and the 3-month average first matched 2019 in February 2023. Of the 6 airlines that flew all of 2019, 3 regained their 2019 level (IndiGo in May 2022; Air India Group in December 2022; Alliance Air in December 2022), but only 2 are still above it. Go First, SpiceJet, TruJet never got back.
2. **Concentration has risen sharply.** IndiGo's share went from 47% to 64% and the market concentration index (HHI) from 3,097 to 4,851 (above 2,500 counts as highly concentrated). Air India Group moved from 23% to 26% and SpiceJet from 15% to 3%.
3. **The index separates the airlines clearly, and the order is robust.** Among airlines scored on all three pillars, IndiGo leads (66) ahead of Air India Group (58), and Alliance Air is last (23). Akasa Air scores highest overall (72) on two pillars, as it has no 2019 level. No airline's rank moves by more than 2 places across four different weightings.
4. **Growth and consistency predict; efficiency does not.** Of the ten metrics, growth momentum lines up best with the next six months of growth (+0.79, partly because growth persists), followed by consistency vs 2019 (+0.63). The Operations pillar scores -0.06 and load-factor change vs 2019 -0.36. The combined index scores +0.09: it describes how an airline is doing today, while growth says where it is heading.
5. **A backtest shows both the use and the limit.** Using only data up to May 2023, Go First ranked 5 of 5 (index 36), dragged down by the weakest momentum pillar. The gap to the next airline was small, and traffic data cannot see finances, so a score like this is a warning light, not a verdict.